In [ ]:
!pip install pytorch-tcn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
import warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from numpy.typing import ArrayLike
from pytorch_tcn.tcn import TCN
from pytorch_tcn.conv import TemporalConv1d
from pytorch_tcn.conv import TemporalConvTranspose1d
from sklearn.metrics import precision_recall_fscore_support

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
#학습/검증 데이터 불러오기
data = pd.read_csv('/Users/yub/Documents/ssu_32/CapstoneDesign1/Datasets/NF_v3_pre.csv',index_col=0)
data.head()

In [ ]:
#세션 별 groupby
label=['IPV4_SRC_ADDR','L4_SRC_PORT','IPV4_DST_ADDR','L4_DST_PORT','PROTOCOL']
grouped = data.groupby(label)

In [ ]:
#총 세션 수
len(grouped)

In [ ]:
grouped_keys = list(grouped.groups.keys())
grouped_keys[:10]

In [ ]:
#세션 길이가 1이 아닌 경우
session_sizes = grouped.size()
non_single_sessions = (session_sizes != 1).sum()
print("전체 세션 수:", len(session_sizes))
print("1개짜리 세션 수:", non_single_sessions)

In [ ]:
#세션 길이가 1인 경우
single_sessions = (session_sizes == 1).sum()
total_sessions = len(session_sizes)
print("전체 세션 수:", total_sessions)
print("1개짜리 세션 수:", single_sessions)

In [ ]:
#세션 길이 1인 경우 공격 수(97712/1804015)
single_session_keys = session_sizes[session_sizes == 1].index

single_session_attacks = data[
    data[label].apply(tuple, axis=1).isin(single_session_keys) & (data['Label'] == 1)]

single_session_attacks

In [ ]:
#세션 길이가 1이 아닌 경우의 공격 수(29981/63833)
non_single_session_keys = session_sizes[session_sizes != 1].index

non_single_session_attacks = data[
    data[label].apply(tuple, axis=1).isin(non_single_session_keys) & (data['Label'] == 1)]

non_single_session_attacks

In [ ]:
#세션(5개 이상)
df_keys = session_sizes[session_sizes >= 1].index

#제거할 feature
drop_col = ['FLOW_START_MILLISECONDS','FLOW_END_MILLISECONDS','IPV4_SRC_ADDR','L4_SRC_PORT','IPV4_DST_ADDR','L4_DST_PORT','PROTOCOL','Label','Attack']
feature_col = [c for c in data.columns if c not in drop_col]

#각 key 별 session
X_list=[]
y_list=[]
for key in df_keys:
  df_group=grouped.get_group(key).sort_values('FLOW_START_MILLISECONDS')
  X = df_group[feature_col].values
  X_list.append(X)
  y = int(df_group['Label'].max())
  y_list.append(y)


In [ ]:
#확인 용도
print("session_key",df_keys[0])
print("X_list",X_list[0])

In [ ]:
#확인 용도
print(y_list)

In [ ]:
for n,i in enumerate(X_list):
  if len(i)>50:
    print(n, len(i))

In [ ]:
#세션의 flow 개수, feature 개수
X_list[0].shape

In [ ]:
#세션 크기 통일시키
#X_list: (세션의 flow 개수, feature 개수) => n*f
#y_list: 각 세션별 label 0/1

max_len = 50  #세션 길이 상한선
F = X_list[0].shape[1] #세션 feature 수

# 세션 길이가 50보다 크면 truncate, 작으면 0으로 padding
def pad_sequence(X, max_len, F):
    L = X.shape[0]

    #세션 길이가 50보다 큰 경우
    if L >= max_len:
        return X[:max_len]
    #세션 길이가 50보다 작은 경우
    out = np.zeros((max_len, F), dtype=np.float32) #n*f padding
    out[:L] = X
    return out

X_arr = np.stack([pad_sequence(x, max_len, F) for x in X_list])  # (N, max_len, F)
y_arr = np.array(y_list, dtype=np.float32)              # y_list => float으로 변환


In [ ]:
#PyTorch Dataset으로 변환, train/val 나눠서 Dataloader
#(Batch(N), Length(F), Channel(max_len))
import torch
from torch.utils.data import Dataset, DataLoader

class SessionDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)  # (N, L, F)
        self.y = torch.tensor(y, dtype=torch.float32)  # (N,)

    #세션 개수(dataset 길이)
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = self.X[idx].permute(1, 0)  # (L, F) -> (F, L)  # NCL
        return x, self.y[idx]

dataset = SessionDataset(X_arr, y_arr)

# train / val 분할 (7:3)
n = len(dataset)
n_train = int(n * 0.7)
n_val = n - n_train

train_ds, val_ds = torch.utils.data.random_split(dataset, [n_train, n_val])

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False)


In [ ]:
import torch.nn as nn
from pytorch_tcn import TCN

class SessionTCN(nn.Module):
    def __init__(self, num_features, num_channels=[64, 64, 64],
                 kernel_size=3, dropout=0.2):
        super().__init__()
        self.tcn = TCN(
            num_inputs=num_features,      # feature 개수
            num_channels=num_channels,    # 각 블록 채널 수
            kernel_size=kernel_size,
            dropout=dropout,
            causal=True,                 # 실시간 탐지를 해야하니까 → TRUE?
            input_shape='NCL',
            use_norm='weight_norm',
            activation='relu',
            use_skip_connections=True
        )
        # 최종 분류를 통한 출력(head)
        # TCN 출력: (N, C_last, L) → 시간축 평균 → Linear(1)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),      # (N, C, L) → (N, C, 1)
            nn.Flatten(),                 # (N, C, 1) → (N, C)
            nn.Linear(num_channels[-1], 1)
        )

    def forward(self, x):  # x: (N, F, L)
        h = self.tcn(x)          # (N, C_last, L)
        out = self.head(h)       # (N, 1)
        return out.squeeze(1)    # (N,)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SessionTCN(num_features=F).to(device) # F=45개


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

train_losses = []
val_losses = []
val_accs = []

for epoch in range(10):
    # F1 계산용 리스트
    y_true_all = []
    y_pred_all = []

    model.train()
    total_loss = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)      # (N, F, L)
        y_batch = y_batch.to(device)      # (N,)

        logits = model(X_batch)           # (N,)
        loss = criterion(logits, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * len(X_batch)

    avg_train_loss = total_loss / len(train_loader.dataset)

    # 간단한 검증
    model.eval()
    total_val_loss = 0
    correct = 0
    total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            total_val_loss += loss.item() * len(X_batch)

            probs = torch.sigmoid(logits)
            preds = (probs > 0.5).float()
            correct += (preds == y_batch).sum().item()
            total += len(y_batch)

            # F1 계산용
            y_true_all.append(y_batch.cpu())
            y_pred_all.append(preds.cpu())

    # 전체 val 결과에 대해 지표 계산
    y_true_all = torch.cat(y_true_all).numpy()
    y_pred_all = torch.cat(y_pred_all).numpy()

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true_all,
        y_pred_all,
        average='binary',
        pos_label=1
    )

    avg_val_loss = total_val_loss / len(val_loader.dataset)
    val_acc = correct / total

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    val_accs.append(val_acc)

    print(
        f"[Epoch {epoch+1}] "
        f"train_loss={avg_train_loss:.4f}  "
        f"val_loss={avg_val_loss:.4f}  "
        f"val_acc={val_acc:.4f}  "
        f"prec(attack)={precision:.4f}  "
        f"recall(attack)={recall:.4f}  "
        f"f1(attack)={f1:.4f}")


In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, 11)

plt.figure(figsize=(10,4))

# 1) Loss
plt.subplot(1, 2, 1)
plt.plot(epochs, train_losses, marker='o', label='Train Loss')
plt.plot(epochs, val_losses, marker='o', label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Train vs Val Loss')
plt.legend()
plt.grid(True)

# 2) Accuracy
plt.subplot(1, 2, 2)
plt.plot(epochs, val_accs, marker='o', label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy')
plt.ylim(0, 1)
plt.grid(True)

plt.tight_layout()
plt.show()
